In [1]:
from sktime.classification.sklearn import RotationForest
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import Generation
from sklearn.model_selection import train_test_split
import scipy.stats as stats
import joblib

In [2]:
def runASPT(PT,lrt):
    a=np.sum(PT,axis=1)!=0
    b=np.sum(np.isnan(PT),axis=1)==0
    c=np.sum(np.isinf(PT),axis=1)==0
    d=np.sum(np.abs(PT)>10**8,axis=1)==0
    e=np.sum(lrt,axis=1)!=0
    
    ind=a & b & c & d & e
    lRT=lrt[ind,:]
    PT=PT[ind,:]
    
    lERT=np.zeros([len(lRT),10])
    for i in range(len(lRT)):
        for j in range(10):
            lERT[i,j]=(np.sum(lRT[i,j*30:(j+1)*30])+np.sum(lRT[i,j*30:(j+1)*30]==-1)*10001)/np.max([np.sum(lRT[i,j*30:(j+1)*30]!=-1),1])
    
    
    
    y = np.argmin(lERT, axis=1)
    label = np.zeros([len(y), 301])
    label[:, 0] = y
    lRT[lRT == -1] = 10000
    label[:, 1:] = lRT
    
    (TrainX, TestX,label1, label2) = train_test_split(PT, label, test_size=0.2,random_state=42)
    TrainY=label1[:,0]
    TrainR=label1[:,1:]
    TestY=label2[:,0]
    TestR=label2[:,1:]
    clf = RotationForest(n_estimators=100,random_state=42)
    clf.fit(TrainX, TrainY)
    y_hat = clf.predict(TestX)
    
    acc=0
    for i in range(len(TestY)):
        if TestY[i]==y_hat[i]:
            acc+=1
        elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
            # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
            acc+=1
    print("可接受率",acc/len(TestY))
    joblib.dump(rf_classifier, 'PT_model.pkl')

In [10]:
# AS-PT  +  RGI
PT=np.zeros([1, 560])
lrt = np.zeros([1, 300])

for i in range(25):
    PT = np.append(PT, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\PT\RGI_Traj-{}.csv".format(i+1), "r"), delimiter=","))[:,1:], axis=0)
    lrt = np.append(lrt, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\MyDataset\Dataset_RT\RT-{}.csv".format(i+1), "r"), delimiter=","))[:,1:], axis=0)
    
a=np.sum(PT,axis=1)!=0
b=np.sum(np.isnan(PT),axis=1)==0
c=np.sum(np.isinf(PT),axis=1)==0
d=np.sum(np.abs(PT)>10**8,axis=1)==0
e=np.sum(lrt,axis=1)!=0

ind=a & b & c & d & e
lRT=lrt[ind,:]
PT=PT[ind,:]

lERT=np.zeros([len(lRT),10])
for i in range(len(lRT)):
    for j in range(10):
        lERT[i,j]=(np.sum(lRT[i,j*30:(j+1)*30])+np.sum(lRT[i,j*30:(j+1)*30]==-1)*10001)/np.max([np.sum(lRT[i,j*30:(j+1)*30]!=-1),1])



y = np.argmin(lERT, axis=1)
label = np.zeros([len(y), 301])
label[:, 0] = y
lRT[lRT == -1] = 10000
label[:, 1:] = lRT

(TrainX, TestX,label1, label2) = train_test_split(PT, label, test_size=0.2,random_state=42)
TrainY=label1[:,0]
TrainR=label1[:,1:]
TestY=label2[:,0]
TestR=label2[:,1:]
clf = RotationForest(n_estimators=100,random_state=42)
clf.fit(TrainX, TrainY)
y_hat = clf.predict(TestX)

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

ValueError: the number of columns changed from 561 to 551 at row 88; use `usecols` to select a subset and avoid this error

In [ ]:
# BBOB
PT=np.zeros([1, 560])
lrt = np.zeros([1, 300])

for i in range(24):
    PT = np.append(PT, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\BBOB_Traj-{}.csv".format(i+1), "r"), delimiter=",")), axis=0)
    lrt = np.append(lrt,np.loadtxt(open("D:\Dataset\Mywork\DataAll\BBOB\\bbob-{}.csv".format(i + 1), "r"),delimiter=",")[:, 300:600], axis=0)
runASPT(PT,lrt)

In [19]:
# Affine
PT=np.zeros([1, 560])
lrt = np.zeros([1, 300])

for i in range(23):
    PT = np.append(PT, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\Affine_Traj-{}.csv".format(i+1), "r"), delimiter=",")), axis=0)
PT=PT[1:,:]
lrt= np.loadtxt(open(r"D:\Dataset\Mywork\DataAll\Affine\affine.csv", "r"), delimiter=",")[:, 300:600]
runASPT(PT,lrt)

可接受率 0.7833333333333333


In [23]:
# Zigzag
PT=np.zeros([1, 560])
Zigzag_rt = np.zeros([1, 300])
for i in range(5):
    for j in range(5):
        PT = np.append(PT, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\Zigzag_Traj-{}-{}.csv".format(i, j), "r"), delimiter=",")), axis=0)
        Zigzag_rt = np.append(Zigzag_rt, np.loadtxt(open("D:\Dataset\Mywork\DataAll\Zigzag\\zigzag-{}-{}.csv".format(i, j), "r"), delimiter=",")[:,300:600], axis=0)
runASPT(PT,Zigzag_rt)

可接受率 1.0


In [64]:
#GPB
PT=np.zeros([1, 560])
GPB_rt = np.zeros([1, 300])
for i in range(5):
    # PT = np.append(PT, pd.read_csv("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\GPBA-Traj-{}.csv".format(i), header=None,delimiter=",").to_numpy(), axis=0)
    PT = np.append(PT, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\GPBA-Traj-{}.csv".format(i), "r"), delimiter=",")), axis=0)
    GPB_rt = np.append(GPB_rt, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfA-{}.csv".format(i), "r"),delimiter=",")[:, 300:600], axis=0)
for i in range(5):
    PT = np.append(PT, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\GPBB-Traj-{}.csv".format(i), "r"), delimiter=",")), axis=0)
    GPB_rt = np.append(GPB_rt, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfB-{}.csv".format(i), "r"),
                                          delimiter=",")[:, 300:600], axis=0)
runASPT(PT,GPB_rt)

D:\Python39\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


可接受率 0.6591928251121076


In [58]:
aa=np.loadtxt("D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\PT\GPBA-Traj-{}.csv".format(i),delimiter=",")

In [51]:
np.sum(aa)

TypeError: can only concatenate str (not "int") to str